# Train final XGBoost classifier

**Stage 2 of 2 — deployable model (not for reporting KPIs).**

| Stage | Notebook | Role |
|-------|----------|------|
| 1 Nested gene CV | `cv_xgboost.ipynb` | Honest outer-fold metrics |
| 2 Single-loop gene CV on **all** labelled data | **this notebook** | One `θ*`, `n_estimators*`, Youden threshold |
| then | | Fit on **all** labelled rows and save artifacts |

Why a second CV? Nested CV yields a *different* hyperparam set per outer fold.
Here we re-tune once on the full labelled table so production has a single coherent
config. The printed **inner mean ROC-AUC** from that tune is a selection diagnostic
only — quote nested-CV outer metrics for performance.

Inference: `notebooks/infer_xgboost.ipynb`.


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
sys.path.insert(0, str(project_root / "src"))

from final_model import (
    DEFAULT_META_PATH,
    DEFAULT_MODEL_PATH,
    train_final_classifier,
)

N_FOLDS = 5
N_TRIALS = 12
SEED = 42
THRESHOLD_METHOD = "youden"

## Single-loop gene CV tune → fit all → persist

Same as `python src/final_model.py`:

1. Assign gene folds on the full labelled table
2. `nested_tune` = **standard (single-loop) gene CV** + random search → best `θ*`,
   `n_estimators*` (median of early-stopped tree counts), Youden threshold
3. Fit XGBoost on **every** labelled row with that fixed tree budget (no holdout)
4. Write `models/xgb_final_pathogenicity.json` + `*_meta.json`


In [3]:
result = train_final_classifier(
    n_folds=N_FOLDS,
    n_trials=N_TRIALS,
    seed=SEED,
    threshold_method=THRESHOLD_METHOD,
    model_path=DEFAULT_MODEL_PATH,
    meta_path=DEFAULT_META_PATH,
)
meta = result["meta"]
pd.Series(meta["params"], name="param").to_frame()

Tuned on 11563 variants / 137 genes
Inner mean ROC-AUC (full-data tune CV): 0.673
n_estimators=79  threshold=0.4894 (youden)
params={'subsample': 0.9, 'reg_lambda': 1.0, 'min_child_weight': 10, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8, 'scale_pos_weight': 0.8203715365239295}
Saved model: /home/milijanas/projects/clinvar-pathogenicity-classifier/models/xgb_final_pathogenicity.json
Saved meta:  /home/milijanas/projects/clinvar-pathogenicity-classifier/models/xgb_final_pathogenicity_meta.json


,param
subsample,0.900000
reg_lambda,1.000000
min_child_weight,10.000000
max_depth,6.000000
learning_rate,0.100000
colsample_bytree,0.800000
scale_pos_weight,0.820372


In [4]:
print(f"model: {result['model_path']}")
print(f"meta:  {result['meta_path']}")
print(f"n_train={meta['n_train']}, n_genes={meta['n_genes']}")
print(f"n_estimators={meta['n_estimators']}, threshold={meta['threshold']:.4f}")
print(f"tune inner mean ROC-AUC={meta['tune_inner_mean_roc_auc']:.3f}")

# result["trials"].sort_values("inner_mean_roc_auc", ascending=False).head(5) # how close runners-up were, whether the pick looks stable

model: /home/milijanas/projects/clinvar-pathogenicity-classifier/models/xgb_final_pathogenicity.json
meta:  /home/milijanas/projects/clinvar-pathogenicity-classifier/models/xgb_final_pathogenicity_meta.json
n_train=11563, n_genes=137
n_estimators=79, threshold=0.4894
tune inner mean ROC-AUC=0.673
